# TabPFNCredit - Comprehensive Results Analysis

This notebook provides comprehensive analysis tools for benchmarking results from the TabPFNCredit project, including:
- **PAMA Analysis**: Probability of Achieving Maximal Accuracy
- **Average Ranks**: Algorithm ranking across datasets
- **Critical Difference Diagrams**: Statistical significance testing (Wilcoxon-Holm)
- **Baseline Improvement**: Percentage improvement over linear baselines
- **Pairwise Comparisons**: Head-to-head algorithm scatter plots
- **Tuning Impact**: Default vs HPO performance analysis

## 1. Setup and Configuration

In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import matplotlib.pyplot as plt
import scipy.stats as stats
from scipy.stats import wilcoxon, friedmanchisquare
import operator
import math
import networkx as nx
from matplotlib.patches import Patch, Rectangle

# =============================================================================
# GLOBAL CONFIGURATION
# =============================================================================

# Experiment to analyze
EXPERIMENT = "experiment1"

# Project root (adjust if running from different location)
PROJECT_ROOT = Path(".").resolve()

# Primary metrics for each task
MAIN_PD_METRIC = "AUC"      # Primary metric for PD (classification)
MAIN_LGD_METRIC = "R2"      # Primary metric for LGD (regression)

# =============================================================================
# ALGORITHM DEFINITIONS
# =============================================================================

# Classical ML Methods
CLASSICAL_METHODS = [
    'XGBoost', 'LightGBM', 'CatBoost', 'RandomForest',
    'LogReg', 'KNN', 'SVM', 'NaiveBayes', 'NCM', 'Dummy'
]

# Deep Learning Methods
DL_METHODS = [
    'TabPFN', 'TabPFNv2', 'TabNet', 'MLP', 'ResNet', 'SAINT',
    'FTTransformer', 'NODE', 'TabTransformer', 'AutoInt', 'DCN2'
]

# All methods (union)
ALL_METHODS = CLASSICAL_METHODS + DL_METHODS

# Baseline algorithms for improvement calculation
PD_BASELINE = 'LogReg'    # Logistic regression for classification
LGD_BASELINE = 'LinearReg' # Linear regression for regression (or LogReg if shared)

print("✅ Configuration loaded")
print(f"   Experiment: {EXPERIMENT}")
print(f"   PD metric: {MAIN_PD_METRIC}")
print(f"   LGD metric: {MAIN_LGD_METRIC}")

## 2. Data Loading

In [ ]:
# =============================================================================
# DATA LOADING - Load summarized results from Summarize_Results.py
# =============================================================================

def load_summarized_results(project_root: Path, experiment: str) -> dict:
    """
    Load all summarized result files into a dictionary.
    
    Returns:
        Dictionary with keys:
        - 'pd_raw': Raw PD results (per fold)
        - 'lgd_raw': Raw LGD results (per fold)
        - 'pd_agg': Aggregated PD results (mean/std per method/dataset/hpo)
        - 'lgd_agg': Aggregated LGD results
        - 'unified': Combined DataFrame in standard analysis format
    """
    summary_dir = project_root / "results" / experiment / "summary"
    
    if not summary_dir.exists():
        raise FileNotFoundError(f"Summary directory not found: {summary_dir}")
    
    print(f"📂 Loading results from: {summary_dir}")
    
    results = {}
    
    # Load raw results (per-fold data)
    pd_raw_file = summary_dir / "summary_pd_raw.csv"
    lgd_raw_file = summary_dir / "summary_lgd_raw.csv"
    
    if pd_raw_file.exists():
        results['pd_raw'] = pd.read_csv(pd_raw_file)
        print(f"   ✅ PD raw: {len(results['pd_raw'])} rows")
    else:
        results['pd_raw'] = pd.DataFrame()
        print(f"   ⚠️ PD raw: Not found")
    
    if lgd_raw_file.exists():
        results['lgd_raw'] = pd.read_csv(lgd_raw_file)
        print(f"   ✅ LGD raw: {len(results['lgd_raw'])} rows")
    else:
        results['lgd_raw'] = pd.DataFrame()
        print(f"   ⚠️ LGD raw: Not found")
    
    # Load aggregated results
    pd_agg_file = summary_dir / "summary_pd_aggregated.csv"
    lgd_agg_file = summary_dir / "summary_lgd_aggregated.csv"
    
    if pd_agg_file.exists():
        results['pd_agg'] = pd.read_csv(pd_agg_file)
        print(f"   ✅ PD aggregated: {len(results['pd_agg'])} rows")
    else:
        results['pd_agg'] = pd.DataFrame()
    
    if lgd_agg_file.exists():
        results['lgd_agg'] = pd.read_csv(lgd_agg_file)
        print(f"   ✅ LGD aggregated: {len(results['lgd_agg'])} rows")
    else:
        results['lgd_agg'] = pd.DataFrame()
    
    # Create unified DataFrame for analysis functions
    results['unified'] = create_unified_dataframe(results)
    
    return results


def create_unified_dataframe(results: dict) -> pd.DataFrame:
    """
    Create a unified DataFrame combining PD and LGD results.
    
    Maps column names to standard format expected by analysis functions:
    - method -> model
    - hpo_mode -> tuning_strategy (NO_HPO -> default, HPO -> optuna)
    - fold_id -> split
    """
    dfs = []
    
    for task in ['pd', 'lgd']:
        raw_key = f'{task}_raw'
        if raw_key not in results or results[raw_key].empty:
            continue
        
        df = results[raw_key].copy()
        
        # Standardize column names
        df = df.rename(columns={
            'method': 'model',
            'fold_id': 'split'
        })
        
        # Map HPO mode to tuning strategy
        df['tuning_strategy'] = df['hpo_mode'].map({
            'NO_HPO': 'default',
            'HPO': 'optuna'
        })
        
        # Ensure task column exists
        if 'task' not in df.columns:
            df['task'] = task
        
        dfs.append(df)
    
    if not dfs:
        return pd.DataFrame()
    
    unified = pd.concat(dfs, ignore_index=True)
    
    print(f"\n📊 Unified DataFrame created:")
    print(f"   Total rows: {len(unified)}")
    print(f"   Tasks: {unified['task'].unique().tolist()}")
    print(f"   Models: {unified['model'].nunique()} unique")
    print(f"   Datasets: {unified['dataset'].nunique()} unique")
    print(f"   Tuning strategies: {unified['tuning_strategy'].unique().tolist()}")
    
    return unified


# Load the data
try:
    RESULTS = load_summarized_results(PROJECT_ROOT, EXPERIMENT)
    UNIFIED_DF = RESULTS['unified']
    print(f"\n✅ Data loading complete!")
except FileNotFoundError as e:
    print(f"\n❌ Error: {e}")
    print("\nPlease run Summarize_Results.py first:")
    print("  python src/postprocessing/Summarize_Results.py --experiment", EXPERIMENT)
    RESULTS = {}
    UNIFIED_DF = pd.DataFrame()

In [ ]:
# =============================================================================
# DATA OVERVIEW
# =============================================================================

if not UNIFIED_DF.empty:
    print("📈 Data Overview:")
    print("=" * 60)
    
    # Task breakdown
    print("\n📊 By Task:")
    for task in UNIFIED_DF['task'].unique():
        task_df = UNIFIED_DF[UNIFIED_DF['task'] == task]
        print(f"   {task.upper()}: {len(task_df)} rows, {task_df['dataset'].nunique()} datasets")
    
    # Tuning strategy breakdown
    print("\n🔧 By Tuning Strategy:")
    for strategy in UNIFIED_DF['tuning_strategy'].unique():
        strategy_df = UNIFIED_DF[UNIFIED_DF['tuning_strategy'] == strategy]
        print(f"   {strategy}: {len(strategy_df)} rows")
    
    # Model list
    print("\n🤖 Available Models:")
    models = sorted(UNIFIED_DF['model'].unique())
    for i, model in enumerate(models, 1):
        print(f"   {i:2d}. {model}")
    
    # Dataset list
    print("\n📁 Available Datasets:")
    for task in UNIFIED_DF['task'].unique():
        datasets = sorted(UNIFIED_DF[UNIFIED_DF['task'] == task]['dataset'].unique())
        print(f"   {task.upper()}: {datasets}")
    
    # Metric columns
    print("\n📏 Available Metrics:")
    numeric_cols = UNIFIED_DF.select_dtypes(include=[np.number]).columns.tolist()
    # Filter out metadata columns
    metric_cols = [c for c in numeric_cols if c not in ['split', 'n_num_features', 'n_cat_features']]
    print(f"   {metric_cols}")
else:
    print("❌ No data loaded. Please check the data loading cell.")

## 3. Helper Functions

In [ ]:
# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def get_available_metrics(df, task=None):
    """Get available metric columns for a given task."""
    # Define expected metrics per task
    pd_metrics = ['AUC', 'Gini', 'KS', 'Brier', 'LogLoss', 'Accuracy', 
                  'Balanced_Accuracy', 'F1', 'Precision', 'Recall', 'MCC']
    lgd_metrics = ['R2', 'RMSE', 'MAE', 'MSE', 'MAPE', 'Correlation', 'Spearman']
    
    if task == 'pd':
        expected = pd_metrics
    elif task == 'lgd':
        expected = lgd_metrics
    else:
        expected = pd_metrics + lgd_metrics
    
    return [m for m in expected if m in df.columns]


def prepare_analysis_data(df, pd_metric=MAIN_PD_METRIC, lgd_metric=MAIN_LGD_METRIC,
                         tuning_strategy=None, task_filter=None):
    """
    Prepare data structure for core analysis functions.
    
    Parameters:
    - df: Input DataFrame (UNIFIED_DF)
    - pd_metric: Metric to use for PD datasets (default: 'AUC')
    - lgd_metric: Metric to use for LGD datasets (default: 'R2')
    - tuning_strategy: Filter by specific tuning strategy ('default', 'optuna', or None)
    - task_filter: Filter by specific task ('pd', 'lgd', or None for both)
    
    Returns:
    - DataFrame with columns: task, dataset, tuning_strategy, model, split, perf_metric
    """
    working_df = df.copy()
    
    # Apply task filter
    if task_filter is not None:
        if task_filter not in ['pd', 'lgd']:
            raise ValueError("task_filter must be 'pd', 'lgd', or None")
        working_df = working_df[working_df['task'] == task_filter]
        print(f"📊 Filtered data for task: {task_filter.upper()}")
    
    # Apply tuning strategy filter
    if tuning_strategy is not None:
        available_tuning = working_df['tuning_strategy'].unique()
        if tuning_strategy not in available_tuning:
            print(f"❌ Warning: tuning_strategy '{tuning_strategy}' not found.")
            print(f"   Available: {list(available_tuning)}")
            return pd.DataFrame()
        working_df = working_df[working_df['tuning_strategy'] == tuning_strategy]
        print(f"📊 Filtered data for tuning strategy: {tuning_strategy}")
    
    # Create performance metric column based on task
    def get_perf_metric(row):
        if row['task'] == 'pd':
            return row.get(pd_metric, np.nan)
        elif row['task'] == 'lgd':
            return row.get(lgd_metric, np.nan)
        return np.nan
    
    working_df['perf_metric'] = working_df.apply(get_perf_metric, axis=1)
    
    # Select relevant columns
    result_cols = ['task', 'dataset', 'tuning_strategy', 'model', 'split', 'perf_metric']
    available_cols = [c for c in result_cols if c in working_df.columns]
    
    result = working_df[available_cols].dropna(subset=['perf_metric'])
    
    print(f"   Prepared {len(result)} data points for analysis")
    print(f"   Models: {sorted(result['model'].unique())}")
    
    return result


# Algorithm display name mapping
ALGORITHM_DISPLAY_NAMES = {
    'tabpfn': 'TabPFN',
    'tabpfnv2': 'TabPFN-v2',
    'xgboost': 'XGBoost',
    'lightgbm': 'LightGBM',
    'catboost': 'CatBoost',
    'randomforest': 'RForest',
    'logreg': 'LogReg',
    'linearreg': 'LinReg',
    'knn': 'KNN',
    'svm': 'SVM',
    'naivebayes': 'NaiveBayes',
    'ncm': 'NCM',
    'dummy': 'Dummy',
    'tabnet': 'TabNet',
    'mlp': 'MLP',
    'resnet': 'ResNet',
    'saint': 'SAINT',
    'fttransformer': 'FT-Trans',
    'node': 'NODE',
}

def get_display_name(algorithm_name):
    """Convert algorithm name to display name."""
    return ALGORITHM_DISPLAY_NAMES.get(algorithm_name.lower(), algorithm_name)


print("✅ Helper functions loaded")

## 4. PAMA Analysis (Probability of Achieving Maximal Accuracy)

In [ ]:
# =============================================================================
# PAMA ANALYSIS
# =============================================================================

def compute_pama_analysis(df, task_filter=None, tuning_strategy=None,
                         pd_metric=MAIN_PD_METRIC, lgd_metric=MAIN_LGD_METRIC,
                         aggregation_level='dataset', top_n=None,
                         plot=True, verbose=True):
    """
    Compute PAMA (Probability of Achieving Maximal Accuracy) analysis.
    
    PAMA measures how often each algorithm achieves the best performance
    across datasets or CV splits.
    
    Parameters:
    - df: Input DataFrame (UNIFIED_DF)
    - task_filter: 'pd', 'lgd', or None for both
    - tuning_strategy: 'default', 'optuna', or None for all
    - aggregation_level: 'dataset' or 'split'
    - top_n: Show only top N algorithms
    - plot: Whether to create visualization
    - verbose: Whether to print detailed results
    
    Returns:
    - DataFrame with PAMA results per algorithm
    """
    if verbose:
        print("=" * 80)
        print("PAMA ANALYSIS - Probability of Achieving Maximal Accuracy")
        print("=" * 80)
    
    # Prepare data
    analysis_data = prepare_analysis_data(
        df, pd_metric=pd_metric, lgd_metric=lgd_metric,
        tuning_strategy=tuning_strategy, task_filter=task_filter
    )
    
    if analysis_data.empty:
        print("❌ No data available for PAMA analysis")
        return pd.DataFrame()
    
    # Get algorithms and datasets
    algorithms = analysis_data['model'].unique()
    datasets = analysis_data['dataset'].unique()
    
    # Calculate wins
    algorithm_wins = {alg: 0.0 for alg in algorithms}
    total_comparisons = 0
    
    for dataset in datasets:
        dataset_data = analysis_data[analysis_data['dataset'] == dataset]
        
        if aggregation_level == 'dataset':
            # Average performance per algorithm on this dataset
            perf = dataset_data.groupby('model')['perf_metric'].mean()
            if len(perf) == 0:
                continue
            
            best_perf = perf.max()
            best_algorithms = perf[perf == best_perf].index.tolist()
            
            win_value = 1.0 / len(best_algorithms)
            for alg in best_algorithms:
                algorithm_wins[alg] += win_value
            total_comparisons += 1
        
        else:  # split level
            splits = dataset_data['split'].unique()
            for split in splits:
                split_data = dataset_data[dataset_data['split'] == split]
                perf = split_data.set_index('model')['perf_metric']
                
                if len(perf) == 0:
                    continue
                
                best_perf = perf.max()
                best_algorithms = perf[perf == best_perf].index.tolist()
                
                win_value = 1.0 / len(best_algorithms)
                for alg in best_algorithms:
                    algorithm_wins[alg] += win_value
                total_comparisons += 1
    
    # Create results DataFrame
    pama_results = []
    for algorithm in algorithms:
        wins = algorithm_wins[algorithm]
        pama_prob = wins / total_comparisons if total_comparisons > 0 else 0.0
        pama_results.append({
            'algorithm': algorithm,
            'wins': wins,
            'total_comparisons': total_comparisons,
            'pama_probability': pama_prob,
            'pama_percentage': pama_prob * 100
        })
    
    pama_df = pd.DataFrame(pama_results)
    pama_df = pama_df.sort_values('pama_probability', ascending=False).reset_index(drop=True)
    
    if verbose:
        comparison_unit = "datasets" if aggregation_level == 'dataset' else "CV splits"
        print(f"\n🏆 PAMA Results ({aggregation_level.upper()} Level):")
        print("-" * 70)
        display_df = pama_df.head(top_n) if top_n else pama_df
        for _, row in display_df.iterrows():
            print(f"{row['algorithm']:<20}: {row['pama_percentage']:6.1f}% "
                  f"({row['wins']:5.1f} wins out of {row['total_comparisons']} {comparison_unit})")
    
    # Plot
    if plot:
        plt.figure(figsize=(12, 8))
        plot_data = pama_df.head(top_n) if top_n else pama_df
        
        colors = ['lightcoral' if i == 0 else 'steelblue' if i < 3 else 'lightblue'
                  for i in range(len(plot_data))]
        
        bars = plt.bar(range(len(plot_data)), plot_data['pama_percentage'],
                       color=colors, alpha=0.8, edgecolor='black', linewidth=1)
        
        plt.xlabel('Algorithm', fontsize=12, fontweight='bold')
        plt.ylabel('PAMA Probability (%)', fontsize=12, fontweight='bold')
        
        task_str = task_filter.upper() if task_filter else "Joint PD+LGD"
        plt.title(f'PAMA Analysis: Probability of Achieving Best Performance\n'
                  f'{task_str} - {aggregation_level.title()} Level', fontsize=14, fontweight='bold')
        
        plt.xticks(range(len(plot_data)), 
                   [get_display_name(a) for a in plot_data['algorithm']], 
                   rotation=45, ha='right')
        plt.grid(True, alpha=0.3, axis='y')
        
        for i, (bar, val) in enumerate(zip(bars, plot_data['pama_percentage'])):
            plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                     f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
        
        plt.tight_layout()
        plt.show()
    
    return pama_df

print("✅ PAMA function loaded")

In [ ]:
# =============================================================================
# PAMA EXAMPLE: PD Classification Results
# =============================================================================

if not UNIFIED_DF.empty:
    print("\n📊 PAMA Analysis: PD Classification (Default Hyperparameters)")
    pama_pd_default = compute_pama_analysis(
        df=UNIFIED_DF,
        task_filter='pd',
        tuning_strategy='default',
        aggregation_level='dataset',
        plot=True,
        verbose=True
    )
else:
    print("❌ No data loaded")

## 5. Average Ranks Analysis

In [ ]:
# =============================================================================
# AVERAGE RANKS ANALYSIS
# =============================================================================

def calculate_average_ranks(df, task_filter=None, tuning_strategy=None,
                           pd_metric=MAIN_PD_METRIC, lgd_metric=MAIN_LGD_METRIC,
                           ranking_level='dataset', plot=True, verbose=True):
    """
    Calculate average ranks for algorithms across datasets or CV splits.
    
    Parameters:
    - df: Input DataFrame
    - task_filter: 'pd', 'lgd', or None
    - tuning_strategy: 'default', 'optuna', or None
    - ranking_level: 'dataset' or 'split'
    - plot: Create visualization
    - verbose: Print detailed results
    
    Returns:
    - DataFrame with average ranks per algorithm
    """
    if verbose:
        print("=" * 80)
        print("AVERAGE RANKS ANALYSIS")
        print("=" * 80)
    
    # Prepare data
    analysis_data = prepare_analysis_data(
        df, pd_metric=pd_metric, lgd_metric=lgd_metric,
        tuning_strategy=tuning_strategy, task_filter=task_filter
    )
    
    if analysis_data.empty:
        print("❌ No data available for ranking analysis")
        return pd.DataFrame()
    
    algorithms = analysis_data['model'].unique()
    datasets = analysis_data['dataset'].unique()
    
    # Collect ranks
    all_ranks = {alg: [] for alg in algorithms}
    
    for dataset in datasets:
        dataset_data = analysis_data[analysis_data['dataset'] == dataset]
        
        if ranking_level == 'dataset':
            # Average performance per algorithm
            perf = dataset_data.groupby('model')['perf_metric'].mean()
            if len(perf) < 2:
                continue
            
            # Rank (1 = best, higher performance = lower rank)
            ranks = perf.rank(ascending=False, method='average')
            for alg in algorithms:
                if alg in ranks.index:
                    all_ranks[alg].append(ranks[alg])
        
        else:  # split level
            splits = dataset_data['split'].unique()
            for split in splits:
                split_data = dataset_data[dataset_data['split'] == split]
                perf = split_data.set_index('model')['perf_metric']
                
                if len(perf) < 2:
                    continue
                
                ranks = perf.rank(ascending=False, method='average')
                for alg in algorithms:
                    if alg in ranks.index:
                        all_ranks[alg].append(ranks[alg])
    
    # Calculate average ranks
    rank_results = []
    for alg in algorithms:
        if all_ranks[alg]:
            rank_results.append({
                'algorithm': alg,
                'avg_rank': np.mean(all_ranks[alg]),
                'std_rank': np.std(all_ranks[alg]),
                'n_rankings': len(all_ranks[alg])
            })
    
    ranks_df = pd.DataFrame(rank_results)
    ranks_df = ranks_df.sort_values('avg_rank').reset_index(drop=True)
    
    if verbose:
        print(f"\n🏆 Average Ranks ({ranking_level.upper()} Level):")
        print("-" * 60)
        for _, row in ranks_df.iterrows():
            print(f"{row['algorithm']:<20}: {row['avg_rank']:.3f} ± {row['std_rank']:.3f}")
    
    # Plot
    if plot and not ranks_df.empty:
        plt.figure(figsize=(12, 8))
        
        y_pos = np.arange(len(ranks_df))
        bars = plt.barh(y_pos, ranks_df['avg_rank'], xerr=ranks_df['std_rank'],
                        color='steelblue', alpha=0.8, edgecolor='black',
                        capsize=5)
        
        plt.yticks(y_pos, [get_display_name(a) for a in ranks_df['algorithm']])
        plt.xlabel('Average Rank (lower is better)', fontsize=12, fontweight='bold')
        plt.ylabel('Algorithm', fontsize=12, fontweight='bold')
        
        task_str = task_filter.upper() if task_filter else "Joint PD+LGD"
        plt.title(f'Average Ranks Analysis\n{task_str} - {ranking_level.title()} Level',
                  fontsize=14, fontweight='bold')
        plt.grid(True, alpha=0.3, axis='x')
        plt.gca().invert_yaxis()  # Best rank at top
        
        plt.tight_layout()
        plt.show()
    
    return ranks_df

print("✅ Average Ranks function loaded")

In [ ]:
# =============================================================================
# AVERAGE RANKS EXAMPLE
# =============================================================================

if not UNIFIED_DF.empty:
    print("\n📊 Average Ranks: PD Classification (Default Hyperparameters)")
    ranks_pd_default = calculate_average_ranks(
        df=UNIFIED_DF,
        task_filter='pd',
        tuning_strategy='default',
        ranking_level='dataset',
        plot=True,
        verbose=True
    )
else:
    print("❌ No data loaded")

## 6. Critical Difference Diagrams (Wilcoxon-Holm Test)

In [ ]:
# =============================================================================
# STATISTICAL SIGNIFICANCE - WILCOXON-HOLM & CD DIAGRAMS
# =============================================================================

def form_cliques(p_values, algorithm_names):
    """
    Form cliques of algorithms that are not significantly different.
    Uses networkx graph theory.
    """
    m = len(algorithm_names)
    g_data = np.zeros((m, m), dtype=np.int64)
    
    for p_val_info in p_values:
        if not p_val_info[3]:  # Not significant
            i = np.where(algorithm_names == p_val_info[0])[0][0]
            j = np.where(algorithm_names == p_val_info[1])[0][0]
            min_i, max_j = min(i, j), max(i, j)
            g_data[min_i, max_j] = 1
    
    graph = nx.Graph(g_data)
    return list(nx.find_cliques(graph))


def wilcoxon_holm_analysis(df_analysis_data, alpha=0.05, verbose=True):
    """
    Apply Wilcoxon signed-rank test with Holm correction.
    
    Returns:
    - p_values: List of (alg1, alg2, p_value, is_significant)
    - average_ranks: Series with algorithm ranks
    - max_datasets: Number of comparison units
    """
    if verbose:
        print("\n🔬 WILCOXON-HOLM STATISTICAL ANALYSIS")
        print("-" * 60)
    
    # Count datasets per algorithm
    alg_counts = df_analysis_data.groupby('model')['dataset'].nunique()
    max_datasets = alg_counts.max()
    complete_algorithms = list(alg_counts[alg_counts == max_datasets].index)
    
    if len(complete_algorithms) < 2:
        print(f"❌ Need at least 2 algorithms with complete coverage")
        return [], pd.Series(), 0
    
    if verbose:
        print(f"   Using {len(complete_algorithms)} algorithms with {max_datasets} datasets")
    
    # Get algorithm performances
    filtered_data = df_analysis_data[df_analysis_data['model'].isin(complete_algorithms)]
    algorithm_performances = {}
    
    for alg in complete_algorithms:
        alg_data = filtered_data[filtered_data['model'] == alg]
        dataset_avg = alg_data.groupby('dataset')['perf_metric'].mean()
        algorithm_performances[alg] = dataset_avg.sort_index().values
    
    # Friedman test
    friedman_stat, friedman_p = friedmanchisquare(*algorithm_performances.values())
    if verbose:
        print(f"   Friedman test: stat={friedman_stat:.4f}, p={friedman_p:.6f}")
    
    # Pairwise Wilcoxon tests
    p_values = []
    m = len(complete_algorithms)
    
    for i in range(m - 1):
        for j in range(i + 1, m):
            alg1, alg2 = complete_algorithms[i], complete_algorithms[j]
            try:
                _, p_val = wilcoxon(algorithm_performances[alg1], 
                                    algorithm_performances[alg2], 
                                    zero_method='pratt')
                p_values.append((alg1, alg2, p_val, False))
            except ValueError:
                continue
    
    # Holm correction
    k = len(p_values)
    if k == 0:
        return [], pd.Series(), 0
    
    p_values.sort(key=lambda x: x[2])
    
    for i in range(k):
        corrected_alpha = alpha / (k - i)
        if p_values[i][2] <= corrected_alpha:
            p_values[i] = (p_values[i][0], p_values[i][1], p_values[i][2], True)
        else:
            break
    
    # Calculate average ranks
    perf_matrix = np.array([algorithm_performances[alg] for alg in complete_algorithms])
    df_ranks = pd.DataFrame(perf_matrix, index=complete_algorithms)
    average_ranks = df_ranks.rank(axis=0, ascending=False).mean(axis=1).sort_values()
    
    if verbose:
        sig_count = sum(1 for p in p_values if p[3])
        print(f"   Significant pairs: {sig_count}/{len(p_values)}")
    
    return p_values, average_ranks, max_datasets


def draw_cd_diagram(average_ranks, p_values, title=None, save_path=None):
    """
    Draw Critical Difference diagram.
    
    Algorithms connected by a horizontal line are NOT significantly different.
    """
    if isinstance(average_ranks, pd.Series):
        ranks_values = average_ranks.values
        names = np.array(average_ranks.index)
    else:
        return
    
    display_names = np.array([get_display_name(n) for n in names])
    
    # Plot dimensions
    k = len(ranks_values)
    width = 10
    textspace = 1.8
    lowv = 1
    highv = max(k, int(np.ceil(max(ranks_values))))
    cline = 0.4
    height = cline + ((k + 1) / 2) * 0.2 + 0.5
    
    fig = plt.figure(figsize=(width, height))
    fig.set_facecolor('white')
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_axis_off()
    
    hf, wf = 1. / height, 1. / width
    
    def rankpos(rank):
        return textspace + (width - 2 * textspace) / (highv - lowv) * (highv - rank)
    
    def line(points, **kwargs):
        ax.plot([wf * p[0] for p in points], [hf * p[1] for p in points], **kwargs)
    
    def text(x, y, s, **kwargs):
        ax.text(wf * x, hf * y, s, **kwargs)
    
    ax.plot([0, 1], [0, 1], c='w')
    ax.set_xlim(0, 1)
    ax.set_ylim(1, 0)
    
    # Main axis line
    line([(textspace, cline), (width - textspace, cline)], color='k', linewidth=2)
    
    # Ticks
    for a in range(lowv, highv + 1):
        line([(rankpos(a), cline - 0.15), (rankpos(a), cline)], color='k', linewidth=2)
        text(rankpos(a), cline - 0.2, str(a), ha='center', va='bottom', size=12)
    
    # Algorithm names and lines
    space = 0.24
    
    for i in range(math.ceil(k / 2)):
        chei = cline + 0.25 + i * space
        line([(rankpos(ranks_values[i]), cline),
              (rankpos(ranks_values[i]), chei),
              (textspace - 0.1, chei)], color='k', linewidth=2)
        text(textspace - 0.2, chei, display_names[i], ha='right', va='center', size=12)
        text(textspace + 0.2, chei - 0.05, f'{ranks_values[i]:.2f}', ha='right', va='center', size=9)
    
    for i in range(math.ceil(k / 2), k):
        chei = cline + 0.25 + (k - i - 1) * space
        line([(rankpos(ranks_values[i]), cline),
              (rankpos(ranks_values[i]), chei),
              (width - textspace + 0.1, chei)], color='k', linewidth=2)
        text(width - textspace + 0.2, chei, display_names[i], ha='left', va='center', size=12)
        text(width - textspace - 0.2, chei - 0.05, f'{ranks_values[i]:.2f}', ha='left', va='center', size=9)
    
    # Draw cliques (non-significant groups)
    cliques = form_cliques(p_values, names)
    start_y = cline + 0.15
    
    for clq in cliques:
        if len(clq) == 1:
            continue
        min_idx, max_idx = min(clq), max(clq)
        line([(rankpos(ranks_values[min_idx]) - 0.02, start_y),
              (rankpos(ranks_values[max_idx]) + 0.02, start_y)],
             color='black', linewidth=4)
        start_y += 0.08
    
    if title:
        plt.title(title, fontsize=12, y=0.95)
    
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
        print(f"📁 Saved: {save_path}")
    
    plt.show()


def statistical_significance_analysis(df, task_filter=None, tuning_strategy=None,
                                      pd_metric=MAIN_PD_METRIC, lgd_metric=MAIN_LGD_METRIC,
                                      aggregation_level='dataset', alpha=0.05,
                                      plot=True, title=None, save_path=None, verbose=True):
    """
    Complete statistical significance analysis with CD diagram.
    
    Returns dictionary with:
    - p_values, average_ranks, significant_pairs, etc.
    """
    if verbose:
        print("=" * 80)
        print("STATISTICAL SIGNIFICANCE ANALYSIS")
        print("=" * 80)
    
    # Prepare data
    analysis_data = prepare_analysis_data(
        df, pd_metric=pd_metric, lgd_metric=lgd_metric,
        tuning_strategy=tuning_strategy, task_filter=task_filter
    )
    
    if analysis_data.empty:
        print("❌ No data available")
        return {}
    
    # Handle aggregation level
    if aggregation_level == 'split':
        stat_data = analysis_data.copy()
        stat_data['comparison_unit'] = stat_data['dataset'] + '_split_' + stat_data['split'].astype(str)
        stat_data = stat_data.rename(columns={'comparison_unit': 'dataset'})
    else:
        stat_data = analysis_data.groupby(['model', 'dataset'])['perf_metric'].mean().reset_index()
    
    # Run analysis
    p_values, average_ranks, max_datasets = wilcoxon_holm_analysis(
        stat_data, alpha=alpha, verbose=verbose
    )
    
    if len(p_values) == 0:
        print("❌ Analysis failed")
        return {}
    
    # Plot CD diagram
    if plot:
        if title is None:
            task_str = task_filter.upper() if task_filter else "Joint PD+LGD"
            title = f'Critical Difference Diagram\n{task_str} - {aggregation_level.title()} Level (α={alpha})'
        draw_cd_diagram(average_ranks, p_values, title=title, save_path=save_path)
    
    results = {
        'p_values': p_values,
        'average_ranks': average_ranks,
        'max_datasets': max_datasets,
        'alpha': alpha,
        'significant_pairs': [(p[0], p[1]) for p in p_values if p[3]],
        'total_comparisons': len(p_values),
        'significant_comparisons': sum(1 for p in p_values if p[3])
    }
    
    if verbose:
        print(f"\n✅ Analysis complete")
        print(f"   Significant pairs: {results['significant_comparisons']}/{results['total_comparisons']}")
    
    return results

print("✅ Statistical significance functions loaded")

In [ ]:
# =============================================================================
# CRITICAL DIFFERENCE EXAMPLE
# =============================================================================

if not UNIFIED_DF.empty:
    print("\n📊 Statistical Significance: PD Classification (Default)")
    cd_results = statistical_significance_analysis(
        df=UNIFIED_DF,
        task_filter='pd',
        tuning_strategy='default',
        aggregation_level='dataset',
        alpha=0.05,
        plot=True,
        verbose=True
    )
else:
    print("❌ No data loaded")

## 7. Baseline Improvement Analysis

In [ ]:
# =============================================================================
# BASELINE IMPROVEMENT ANALYSIS
# =============================================================================

def compute_baseline_improvement(df, task_filter=None, tuning_strategy=None,
                                pd_metric=MAIN_PD_METRIC, lgd_metric=MAIN_LGD_METRIC,
                                baseline_algorithm='LogReg', aggregation_level='dataset',
                                plot=True, figsize=(14, 8), verbose=True):
    """
    Compute percentage improvement of algorithms over a baseline.
    
    Parameters:
    - baseline_algorithm: Algorithm to use as baseline (default: 'LogReg')
    - aggregation_level: 'dataset' or 'split'
    
    Returns:
    - Dictionary with algorithm names as keys and improvement lists as values
    """
    if verbose:
        print("=" * 80)
        print("BASELINE IMPROVEMENT ANALYSIS")
        print("=" * 80)
        print(f"   Baseline: {baseline_algorithm}")
    
    # Prepare data
    analysis_data = prepare_analysis_data(
        df, pd_metric=pd_metric, lgd_metric=lgd_metric,
        tuning_strategy=tuning_strategy, task_filter=task_filter
    )
    
    if analysis_data.empty:
        print("❌ No data available")
        return {}
    
    # Check baseline exists
    available_models = analysis_data['model'].unique()
    if baseline_algorithm not in available_models:
        print(f"❌ Baseline '{baseline_algorithm}' not found")
        print(f"   Available: {sorted(available_models)}")
        return {}
    
    datasets = analysis_data['dataset'].unique()
    other_algorithms = [a for a in available_models if a != baseline_algorithm]
    
    # Calculate improvements
    improvements = {alg: [] for alg in other_algorithms}
    
    for dataset in datasets:
        dataset_data = analysis_data[analysis_data['dataset'] == dataset]
        
        if aggregation_level == 'dataset':
            perf = dataset_data.groupby('model')['perf_metric'].mean()
            if baseline_algorithm not in perf.index:
                continue
            
            baseline_perf = perf[baseline_algorithm]
            
            for alg in other_algorithms:
                if alg in perf.index and baseline_perf != 0:
                    pct_imp = ((perf[alg] - baseline_perf) / abs(baseline_perf)) * 100
                    improvements[alg].append(pct_imp)
        
        else:  # split level
            splits = dataset_data['split'].unique()
            for split in splits:
                split_data = dataset_data[dataset_data['split'] == split]
                perf = split_data.set_index('model')['perf_metric']
                
                if baseline_algorithm not in perf.index:
                    continue
                
                baseline_perf = perf[baseline_algorithm]
                
                for alg in other_algorithms:
                    if alg in perf.index and baseline_perf != 0:
                        pct_imp = ((perf[alg] - baseline_perf) / abs(baseline_perf)) * 100
                        improvements[alg].append(pct_imp)
    
    # Filter algorithms with data
    improvements = {k: v for k, v in improvements.items() if v}
    
    if not improvements:
        print("❌ No improvements calculated")
        return {}
    
    # Sort by median improvement
    sorted_algs = sorted(improvements.keys(), 
                         key=lambda x: np.median(improvements[x]), 
                         reverse=True)
    
    if verbose:
        print(f"\n📊 Improvement Summary (vs {baseline_algorithm}):")
        print("-" * 60)
        for alg in sorted_algs:
            vals = improvements[alg]
            print(f"{alg:<20}: median={np.median(vals):+.2f}%, "
                  f"mean={np.mean(vals):+.2f}%, n={len(vals)}")
    
    # Plot boxplot
    if plot:
        plt.figure(figsize=figsize)
        
        data = [improvements[alg] for alg in sorted_algs]
        labels = [get_display_name(alg) for alg in sorted_algs]
        
        bp = plt.boxplot(data, labels=labels, notch=True, patch_artist=True)
        
        # Color boxes
        for i, patch in enumerate(bp['boxes']):
            median = np.median(data[i])
            if median > 0:
                patch.set_facecolor('lightgreen')
            else:
                patch.set_facecolor('lightcoral')
            patch.set_alpha(0.7)
        
        plt.axhline(y=0, color='red', linestyle='--', linewidth=2, alpha=0.7, label='Baseline')
        
        plt.xlabel('Algorithm', fontsize=12, fontweight='bold')
        plt.ylabel(f'% Improvement over {baseline_algorithm}', fontsize=12, fontweight='bold')
        
        task_str = task_filter.upper() if task_filter else "Joint PD+LGD"
        plt.title(f'Baseline Improvement Analysis\n{task_str} - {aggregation_level.title()} Level',
                  fontsize=14, fontweight='bold')
        
        plt.xticks(rotation=45, ha='right')
        plt.grid(True, alpha=0.3, axis='y')
        plt.legend(loc='upper right')
        
        plt.tight_layout()
        plt.show()
    
    return improvements

print("✅ Baseline improvement function loaded")

In [ ]:
# =============================================================================
# BASELINE IMPROVEMENT EXAMPLE
# =============================================================================

if not UNIFIED_DF.empty:
    # Find a suitable baseline
    available_models = UNIFIED_DF['model'].unique()
    print(f"Available models: {sorted(available_models)}")
    
    # Try to find LogReg or similar baseline
    baseline = None
    for candidate in ['LogReg', 'LinearReg', 'Dummy', 'KNN']:
        if candidate in available_models:
            baseline = candidate
            break
    
    if baseline:
        print(f"\n📊 Baseline Improvement: PD Classification (vs {baseline})")
        improvements = compute_baseline_improvement(
            df=UNIFIED_DF,
            task_filter='pd',
            tuning_strategy='default',
            baseline_algorithm=baseline,
            aggregation_level='dataset',
            plot=True,
            verbose=True
        )
    else:
        print("❌ No suitable baseline found in data")
else:
    print("❌ No data loaded")

## 8. Tuning Impact Analysis

In [ ]:
# =============================================================================
# TUNING IMPACT ANALYSIS
# =============================================================================

def analyze_tuning_impact(df, task_filter=None, pd_metric=MAIN_PD_METRIC, 
                         lgd_metric=MAIN_LGD_METRIC, plot=True, verbose=True):
    """
    Compare algorithm performance: Default vs Optimized (HPO).
    
    Creates overlapping bar chart showing performance gain from HPO.
    """
    if verbose:
        print("=" * 80)
        print("TUNING IMPACT ANALYSIS: Default vs HPO")
        print("=" * 80)
    
    # Prepare data for both strategies
    metric = pd_metric if task_filter == 'pd' else lgd_metric
    
    default_data = prepare_analysis_data(
        df, pd_metric=pd_metric, lgd_metric=lgd_metric,
        task_filter=task_filter, tuning_strategy='default'
    )
    
    hpo_data = prepare_analysis_data(
        df, pd_metric=pd_metric, lgd_metric=lgd_metric,
        task_filter=task_filter, tuning_strategy='optuna'
    )
    
    if default_data.empty or hpo_data.empty:
        print("❌ Missing data for one or both tuning strategies")
        return {}
    
    # Get common algorithms
    common_algs = set(default_data['model'].unique()) & set(hpo_data['model'].unique())
    
    if not common_algs:
        print("❌ No common algorithms")
        return {}
    
    # Calculate average performance
    results = {}
    for alg in common_algs:
        default_perf = default_data[default_data['model'] == alg]['perf_metric'].mean()
        hpo_perf = hpo_data[hpo_data['model'] == alg]['perf_metric'].mean()
        results[alg] = {
            'default': default_perf,
            'hpo': hpo_perf,
            'improvement': hpo_perf - default_perf,
            'improvement_pct': ((hpo_perf - default_perf) / abs(default_perf)) * 100 if default_perf != 0 else 0
        }
    
    # Sort by HPO performance
    sorted_algs = sorted(results.keys(), key=lambda x: results[x]['hpo'], reverse=True)
    
    if verbose:
        print(f"\n📊 Tuning Impact Summary:")
        print("-" * 70)
        for alg in sorted_algs:
            r = results[alg]
            print(f"{alg:<20}: Default={r['default']:.4f}, HPO={r['hpo']:.4f}, "
                  f"Δ={r['improvement']:+.4f} ({r['improvement_pct']:+.1f}%)")
    
    # Plot
    if plot:
        fig, ax = plt.subplots(figsize=(14, 8))
        
        x_pos = np.arange(len(sorted_algs))
        default_vals = [results[a]['default'] for a in sorted_algs]
        hpo_vals = [results[a]['hpo'] for a in sorted_algs]
        
        # Overlapping bars
        bars_default = ax.bar(x_pos, default_vals, width=0.8, label='Default',
                              color='lightcoral', alpha=0.9, edgecolor='darkred', linewidth=1.5)
        bars_hpo = ax.bar(x_pos, hpo_vals, width=0.7, label='HPO (Optuna)',
                          color='steelblue', alpha=0.8, edgecolor='darkblue', linewidth=1.5)
        
        ax.set_xlabel('Algorithm', fontsize=12, fontweight='bold')
        ax.set_ylabel(f'Average {metric}', fontsize=12, fontweight='bold')
        
        task_str = task_filter.upper() if task_filter else "Joint PD+LGD"
        ax.set_title(f'Tuning Impact: Default vs HPO Performance\n{task_str}',
                     fontsize=14, fontweight='bold')
        
        ax.set_xticks(x_pos)
        ax.set_xticklabels([get_display_name(a) for a in sorted_algs], rotation=45, ha='right')
        ax.legend(loc='upper right', fontsize=12)
        ax.grid(axis='y', alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    
    return results

print("✅ Tuning impact function loaded")

In [ ]:
# =============================================================================
# TUNING IMPACT EXAMPLE
# =============================================================================

if not UNIFIED_DF.empty:
    # Check if both tuning strategies exist
    strategies = UNIFIED_DF['tuning_strategy'].unique()
    
    if 'default' in strategies and 'optuna' in strategies:
        print("\n📊 Tuning Impact: PD Classification")
        tuning_results = analyze_tuning_impact(
            df=UNIFIED_DF,
            task_filter='pd',
            plot=True,
            verbose=True
        )
    else:
        print(f"⚠️ Both 'default' and 'optuna' strategies needed. Found: {strategies}")
else:
    print("❌ No data loaded")

## 9. Pairwise Algorithm Comparison

In [ ]:
# =============================================================================
# PAIRWISE COMPARISON SCATTER PLOT
# =============================================================================

def compare_algorithms_scatter(df, algorithm1, algorithm2, task_filter=None,
                               tuning_strategy=None, pd_metric=MAIN_PD_METRIC,
                               lgd_metric=MAIN_LGD_METRIC, aggregation_level='dataset',
                               plot=True, verbose=True):
    """
    Create scatter plot comparing two algorithms.
    
    Points above diagonal: algorithm2 is better
    Points below diagonal: algorithm1 is better
    """
    if verbose:
        print("=" * 80)
        print(f"PAIRWISE COMPARISON: {algorithm1} vs {algorithm2}")
        print("=" * 80)
    
    # Prepare data
    analysis_data = prepare_analysis_data(
        df, pd_metric=pd_metric, lgd_metric=lgd_metric,
        tuning_strategy=tuning_strategy, task_filter=task_filter
    )
    
    if analysis_data.empty:
        print("❌ No data available")
        return pd.DataFrame()
    
    # Check algorithms exist
    available = analysis_data['model'].unique()
    for alg in [algorithm1, algorithm2]:
        if alg not in available:
            print(f"❌ Algorithm '{alg}' not found. Available: {sorted(available)}")
            return pd.DataFrame()
    
    # Build comparison data
    alg1_data = analysis_data[analysis_data['model'] == algorithm1]
    alg2_data = analysis_data[analysis_data['model'] == algorithm2]
    
    comparison_rows = []
    datasets = set(alg1_data['dataset'].unique()) & set(alg2_data['dataset'].unique())
    
    for dataset in datasets:
        if aggregation_level == 'dataset':
            perf1 = alg1_data[alg1_data['dataset'] == dataset]['perf_metric'].mean()
            perf2 = alg2_data[alg2_data['dataset'] == dataset]['perf_metric'].mean()
            comparison_rows.append({
                'dataset': dataset,
                'alg1_perf': perf1,
                'alg2_perf': perf2,
                'winner': algorithm2 if perf2 > perf1 else algorithm1 if perf1 > perf2 else 'tie'
            })
        else:  # split level
            alg1_ds = alg1_data[alg1_data['dataset'] == dataset]
            alg2_ds = alg2_data[alg2_data['dataset'] == dataset]
            splits = set(alg1_ds['split'].unique()) & set(alg2_ds['split'].unique())
            
            for split in splits:
                perf1 = alg1_ds[alg1_ds['split'] == split]['perf_metric'].iloc[0]
                perf2 = alg2_ds[alg2_ds['split'] == split]['perf_metric'].iloc[0]
                comparison_rows.append({
                    'dataset': f"{dataset}_s{split}",
                    'alg1_perf': perf1,
                    'alg2_perf': perf2,
                    'winner': algorithm2 if perf2 > perf1 else algorithm1 if perf1 > perf2 else 'tie'
                })
    
    if not comparison_rows:
        print("❌ No comparison data")
        return pd.DataFrame()
    
    comparison_df = pd.DataFrame(comparison_rows)
    
    # Calculate statistics
    total = len(comparison_df)
    alg1_wins = len(comparison_df[comparison_df['winner'] == algorithm1])
    alg2_wins = len(comparison_df[comparison_df['winner'] == algorithm2])
    ties = total - alg1_wins - alg2_wins
    
    if verbose:
        print(f"\n📊 Results: {total} comparisons")
        print(f"   {algorithm1} wins: {alg1_wins} ({100*alg1_wins/total:.1f}%)")
        print(f"   {algorithm2} wins: {alg2_wins} ({100*alg2_wins/total:.1f}%)")
        print(f"   Ties: {ties} ({100*ties/total:.1f}%)")
    
    # Plot
    if plot:
        plt.figure(figsize=(10, 10))
        
        colors = ['steelblue' if w == algorithm2 else 'lightcoral' if w == algorithm1 else 'gray'
                  for w in comparison_df['winner']]
        
        plt.scatter(comparison_df['alg1_perf'], comparison_df['alg2_perf'],
                    c=colors, alpha=0.7, s=60, edgecolors='black', linewidth=0.5)
        
        # Diagonal line
        min_val = min(comparison_df['alg1_perf'].min(), comparison_df['alg2_perf'].min())
        max_val = max(comparison_df['alg1_perf'].max(), comparison_df['alg2_perf'].max())
        plt.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.5, linewidth=2)
        
        plt.xlabel(f'{get_display_name(algorithm1)} Performance', fontsize=12, fontweight='bold')
        plt.ylabel(f'{get_display_name(algorithm2)} Performance', fontsize=12, fontweight='bold')
        
        task_str = task_filter.upper() if task_filter else "Joint PD+LGD"
        plt.title(f'Algorithm Comparison: {get_display_name(algorithm1)} vs {get_display_name(algorithm2)}\n'
                  f'{task_str} - {aggregation_level.title()} Level',
                  fontsize=14, fontweight='bold')
        
        # Legend
        legend_elements = [
            Patch(facecolor='steelblue', alpha=0.7, label=f'{get_display_name(algorithm2)} wins ({alg2_wins})'),
            Patch(facecolor='lightcoral', alpha=0.7, label=f'{get_display_name(algorithm1)} wins ({alg1_wins})'),
            plt.Line2D([0], [0], color='k', linestyle='--', alpha=0.5, label='Equal')
        ]
        plt.legend(handles=legend_elements, loc='upper left')
        
        plt.grid(True, alpha=0.3)
        plt.axis('equal')
        plt.tight_layout()
        plt.show()
    
    return comparison_df

print("✅ Pairwise comparison function loaded")

In [ ]:
# =============================================================================
# PAIRWISE COMPARISON EXAMPLE
# =============================================================================

if not UNIFIED_DF.empty:
    available = sorted(UNIFIED_DF['model'].unique())
    print(f"Available algorithms: {available}")
    
    if len(available) >= 2:
        # Compare first two available algorithms
        alg1, alg2 = available[0], available[1]
        
        print(f"\n📊 Comparing: {alg1} vs {alg2}")
        comparison = compare_algorithms_scatter(
            df=UNIFIED_DF,
            algorithm1=alg1,
            algorithm2=alg2,
            task_filter='pd',
            tuning_strategy='default',
            aggregation_level='split',
            plot=True,
            verbose=True
        )
else:
    print("❌ No data loaded")

## 10. Quick Summary

In [ ]:
# =============================================================================
# ANALYSIS SUMMARY
# =============================================================================

print("=" * 80)
print(" TabPFNCredit Analysis Notebook Summary")
print("=" * 80)

print("""
📊 AVAILABLE FUNCTIONS:

1. compute_pama_analysis(df, task_filter, tuning_strategy, aggregation_level)
   → Probability of Achieving Maximal Accuracy
   → Shows which algorithms most often achieve best performance

2. calculate_average_ranks(df, task_filter, tuning_strategy, ranking_level)
   → Average ranking across datasets/splits
   → Lower rank = better overall performance

3. statistical_significance_analysis(df, task_filter, tuning_strategy, alpha)
   → Wilcoxon-Holm test with CD diagram
   → Connected algorithms are NOT significantly different

4. compute_baseline_improvement(df, task_filter, baseline_algorithm)
   → Percentage improvement over baseline (e.g., LogReg)
   → Boxplot visualization

5. analyze_tuning_impact(df, task_filter)
   → Compare Default vs HPO performance
   → Overlapping bar chart

6. compare_algorithms_scatter(df, algorithm1, algorithm2, task_filter)
   → Head-to-head scatter plot
   → Points above diagonal: algorithm2 wins

📋 COMMON PARAMETERS:
   - task_filter: 'pd', 'lgd', or None (both)
   - tuning_strategy: 'default', 'optuna', or None (both)
   - aggregation_level/ranking_level: 'dataset' or 'split'

💡 TIPS:
   - Use 'split' level for more statistical power (5x more comparisons)
   - Use 'dataset' level for traditional analysis (more conservative)
   - Check task='pd' and task='lgd' separately before joint analysis
""")

if not UNIFIED_DF.empty:
    print("\n✅ Data loaded successfully!")
    print(f"   {len(UNIFIED_DF)} total rows")
    print(f"   {UNIFIED_DF['model'].nunique()} algorithms")
    print(f"   {UNIFIED_DF['dataset'].nunique()} datasets")
else:
    print("\n⚠️ No data loaded. Run Summarize_Results.py first.")